# FaceForensics++, then a test on DFDC

Trains on FaceForensics++ (c23) with its official splits, reports the test set per manipulation
method, then scores all 400 DFDC sample clips, a dataset the model never trained on.

Before running:

1. **Add Input**: a FaceForensics++ c23 dataset (videos in `original` and per-method folders).
   For the DFDC test, also add the notebook whose output is `dfdc_sample.tar`; without it the
   DFDC part is skipped.
2. **Settings**: Accelerator GPU T4 x2, Internet on.
3. Run the first four cells by hand to check the counts, then **Save Version → Save & Run All**.

In [ ]:
BRANCH = "v2"
!rm -rf /kaggle/working/repo
!git clone -q --depth 1 -b $BRANCH https://github.com/Marwankhater/DEEP-FAKE-DETECTION.git /kaggle/working/repo
!pip install -q "mtcnn>=1.0" lz4 pyyaml

In [ ]:
import glob, os, sys
os.chdir("/kaggle/working/repo")
sys.path.insert(0, "src")
import tensorflow as tf
print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

## 1. What is attached

In [ ]:
from collections import Counter
from pathlib import Path

clips = [Path(p) for p in glob.glob("/kaggle/input/**/*.mp4", recursive=True)]
print(len(clips), "mp4 files under /kaggle/input")
print(Counter(p.parent.as_posix().replace("/kaggle/input/", "") for p in clips).most_common(20))

for tar in glob.glob("/kaggle/input/**/dfdc_sample.tar", recursive=True):
    !tar -xf "$tar" -C /kaggle/working/
print("DFDC sample for the cross-dataset test:", os.path.exists("/kaggle/working/train_sample_videos/metadata.json"))

In [ ]:
from deepfake_detector.config import load_config
from deepfake_detector.extract import video_table
from deepfake_detector.metadata import describe

config = load_config("configs/kaggle_faceforensics.yaml")
if not os.path.exists("/kaggle/working/train_sample_videos/metadata.json"):
    config.data.sources = [s for s in config.data.sources if s["type"] != "dfdc"]
videos = video_table(config)
print(describe(videos))
videos.groupby(["split", "method"]).size().unstack(fill_value=0)

## 2. Extract faces

In [ ]:
from deepfake_detector.extract import extract
index = extract(config)
index.groupby(["split", "label"]).size().unstack()

## 3. Train

In [ ]:
from deepfake_detector.train import train
model_path = train(config)

## 4. Evaluate

In [ ]:
from IPython.display import Image, Markdown, display
from deepfake_detector.evaluate import evaluate

results = evaluate(config)
display(Markdown(open(f"{config.run_dir}/report.md").read()))
display(Image(f"{config.run_dir}/roc.png"), Image(f"{config.run_dir}/confusion.png"))

In [ ]:
!cd /kaggle/working && zip -q -r results_ff.zip runs/ff_efficientnet_b0 faces_ff/videos.csv
print("Download /kaggle/working/results_ff.zip from the Output tab.")